## Initializing polymer systems using a DPD potential
This notebook demonstrates how to use HOOMD DPD potential on a random walk packed polymer box. This achieves a intial frame to start a dense polymer melt simulation with another forcefield.

In [1]:
import matplotlib
import numpy as np  
import gsd, gsd.hoomd 
import hoomd 
import time
import freud
import matplotlib_inline
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.style.use("ggplot")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

## Core Functions

In [2]:
def initialize_snapshot_rand_walk(num_pol, num_mon, density=0.85, bond_length=1.0, buffer=0.1):
    """Create a HOOMD snapshot containing random-walk polymer chains.

    Parameters
    ----------
    num_pol : int
        Number of polymer chains in the simulation.

    num_mon : int
        Number of monomers per polymer chain.

    density : float, optional, default 0.85
        Target particle number density used to determine
        the cubic simulation box size.

    bond_length : float, optional, default 1.0
        Bond length between connected monomers.

    buffer : float, optional, default 0.1
        This buffer is reserved to reduce the size of the box length for the random walk.

    Returns
    -------
    gsd.hoomd.Frame
        HOOMD snapshot containing particle positions,
        bond connectivity, particle types, and box
        dimensions.

    """
    N = num_pol * num_mon
    L = np.cbrt(N / density)  # Calculate box size based on density
    positions = np.zeros((N, 3))
    for i in range(num_pol):
        start = i * num_mon
        positions[start] = np.random.uniform(low=(-L/2),high=(L/2),size=3)
        for j in range(num_mon - 1):
            delta = np.random.uniform(low=(-bond_length/2),high=(bond_length/2),size=3)
            delta /= np.linalg.norm(delta)*bond_length
            positions[start+j+1] = positions[start+j] + delta
    positions = pbc(positions,[L,L,L])
    bonds = []
    for i in range(num_pol):
        start = i * num_mon
        for j in range(num_mon - 1):
            bonds.append([start + j, start + j + 1])
    bonds = np.array(bonds)
    frame = gsd.hoomd.Frame()
    frame.particles.types = ['A']
    frame.particles.N = N
    frame.particles.position = positions
    frame.bonds.N = len(bonds)
    frame.bonds.group = bonds
    frame.bonds.types = ['b']
    frame.configuration.box = [L, L, L, 0, 0, 0]
    return frame

def pbc(d,box):
    """Apply periodic boundary conditions to particle positions.

    Parameters
    ----------
    d : np.ndarray
        Particle position array with shape (N, 3).

    box : list[float]
        Simulation box lengths in the x, y, and z
        directions.

    Returns
    -------
    np.ndarray
        Particle positions wrapped into the simulation box.

    """
    for i in range(3):
        a = d[:,i]
        pos_max = np.max(a)
        pos_min = np.min(a)
        while pos_max > box[i]/2 or pos_min < -box[i]/2:
            a[a < -box[i]/2] += box[i]
            a[a >  box[i]/2] -= box[i]
            pos_max = np.max(a)
            pos_min = np.min(a)
    return d


def check_bond_length_equilibration(snap,num_mon,num_pol,max_bond_length=1.1,min_bond_length=0.95):
    """Check whether all bond lengths are within target limits.

    Parameters
    ----------
    snap : gsd.hoomd.Frame
        HOOMD snapshot containing particle positions
        and box information.

    num_mon : int
        Number of monomers per polymer chain.

    num_pol : int
        Number of polymer chains in the system.

    max_bond_length : float, optional, default 1.1
        Maximum allowable bond length.

    min_bond_length : float, optional, default 0.95
        Minimum allowable bond length.

    Returns
    -------
    bool
        True if all bond lengths are within the
        specified range, otherwise False.

    """
    frame_ds = []
    for j in range(num_pol):
        idx = j*num_mon
        d1 = snap.particles.position[idx:idx+num_mon-1] - snap.particles.position[idx+1:idx+num_mon]
        L = snap.configuration.box[0]
        d1 -= L*np.round(d1/L)
        bond_l = np.linalg.norm(d1,axis=1)
        frame_ds.append(bond_l)
    max_frame_bond_l = np.max(np.array(frame_ds))
    min_frame_bond_l = np.min(np.array(frame_ds))
    print("max: ",max_frame_bond_l," min: ",min_frame_bond_l)
    if max_frame_bond_l <= max_bond_length and min_frame_bond_l >= min_bond_length:
        print("Bonds relaxed.")
        return True
    if max_frame_bond_l > max_bond_length or min_frame_bond_l < min_bond_length:
        return False

def check_inter_particle_distance(snap,minimum_distance=0.95):
    """Check for particle overlaps using neighbor searching.

    Parameters
    ----------
    snap : gsd.hoomd.Frame
        HOOMD snapshot containing particle positions
        and box information.

    minimum_distance : float, optional, default 0.95
        Minimum allowed inter-particle separation.

    Returns
    -------
    bool
        True if no particles are found within the
        specified minimum distance, otherwise False.

    """
    positions = snap.particles.position
    box = snap.configuration.box
    aq = freud.locality.AABBQuery(box,positions)
    aq_query = aq.query(
        query_points=positions,
        query_args=dict(r_min=0.0, r_max=minimum_distance, exclude_ii=True),
    )
    nlist = aq_query.toNeighborList()
    if len(nlist)==0:
        print("Inter-particle separation reached.")
        return True
    else:
        return False

def run_one(A=1000,gamma=1000,k=1000,num_pol=100,num_mon=10,kT=1.0,r_cut = 1.15,bond_l=1.0,dt=0.001,density=0.8,particle_spacing = 1.1
):
    """Run a single polymer equilibration simulation in HOOMD-blue.

    Parameters
    ----------
    A : float, optional, default 1000
        Conservative force parameter for the DPD interaction.

    gamma : float, optional, default 1000
        Dissipative force coefficient for the DPD interaction.

    k : float, optional, default 1000
        Harmonic bond spring constant.

    num_pol : int, optional, default 100
        Number of polymer chains in the simulation.

    num_mon : int, optional, default 10
        Number of monomers per polymer chain.

    kT : float, optional, default 1.0
        Thermal energy used by the DPD thermostat.

    r_cut : float, optional, default 1.15
        Cutoff radius for DPD interactions.

    bond_l : float, optional, default 1.0
        Equilibrium bond length for harmonic bonds.

    dt : float, optional, default 0.001
        Simulation integration timestep.

    density : float, optional, default 0.8
        Target particle number density.

    particle_spacing : float, optional, default 1.1
        Maximum acceptable bond length during equilibration.

    Returns
    -------
    tuple[int, float]
        Tuple containing:
        - total number of particles
        - elapsed runtime in seconds

        If equilibration fails due to timeout,
        runtime is returned as 0.

    """
    print(num_pol*num_mon)
    print(f"\nRunning with A={A}, gamma={gamma}, k={k}, "
          f"num_pol={num_pol}, num_mon={num_mon}")
    start_time = time.perf_counter()
    frame = initialize_snapshot_rand_walk(num_pol, num_mon, density=density)
    build_stop = time.perf_counter()
    print("Total build time: ", build_stop-start_time)
    harmonic = hoomd.md.bond.Harmonic()
    harmonic.params["b"] = dict(r0=bond_l, k=k)
    integrator = hoomd.md.Integrator(dt=dt)
    integrator.forces.append(harmonic)
    simulation = hoomd.Simulation(device=hoomd.device.auto_select(), seed=np.random.randint(65535))
    simulation.operations.integrator = integrator 
    simulation.create_state_from_snapshot(frame)
    const_vol = hoomd.md.methods.ConstantVolume(filter=hoomd.filter.All())
    integrator.methods.append(const_vol)
    nlist = hoomd.md.nlist.Cell(buffer=0.4)
    simulation.operations.nlist = nlist
    DPD = hoomd.md.pair.DPD(nlist, default_r_cut=r_cut, kT=kT)
    DPD.params[('A', 'A')] = dict(A=A, gamma=gamma)
    integrator.forces.append(DPD)
    
    simulation.run(0)
    simulation.run(1000)
    snap=simulation.state.get_snapshot()
    N = num_pol*num_mon
    time_factor = N/90000
    
    while not check_bond_length_equilibration(snap,num_mon, num_pol,max_bond_length=particle_spacing): 
        check_time = time.perf_counter()
        if (check_time-start_time) > 60*time_factor:
            return num_pol*num_mon, 0
        simulation.run(1000)
        snap=simulation.state.get_snapshot()

    while not check_inter_particle_distance(snap,minimum_distance=0.95):
        check_time = time.perf_counter()
        if (check_time-start_time) > 60*time_factor:
            return num_pol*num_mon, 0
        simulation.run(1000)
        snap=simulation.state.get_snapshot()
        
    end_time = time.perf_counter()
    return num_pol*num_mon, end_time - start_time

In [3]:
N,s = run_one(A=1000,gamma=800,k=20000,num_pol=10,num_mon=1000)
print(f"Finished in time = {s:.2f}s")

10000

Running with A=1000, gamma=800, k=20000, num_pol=10, num_mon=1000
Total build time:  0.055446250014938414
max:  1.028181237099016  min:  0.9641677772359297
Bonds relaxed.
Inter-particle separation reached.
Finished in time = 3.32s
